In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [1]:
# ===============================================================
# Actualización de datos SECOP - Trimestral
# ===============================================================

import requests
import pandas as pd
from datetime import datetime, timedelta
import os

# Directorio donde ya tienes los .parquet trimestrales
directorio_salida = r"G:\Mi unidad\DatosAPI_Trimestral_Parquet"
dataset_id = "p6dx-8zbt"
url = f"https://www.datos.gov.co/resource/{dataset_id}.json"


# 5 registros para inspeccionar
resp = requests.get(url)
resp.raise_for_status()
df_preview = pd.DataFrame(resp.json())
print("\n Conexión exitosa a la API.")
print(f" Total de columnas en la base: {len(df_preview.columns)}")
print("\n Nombres de las columnas:")
for i, col in enumerate(df_preview.columns, 1):
    print(f"{i}. {col}")



 Conexión exitosa a la API.
 Total de columnas en la base: 57

 Nombres de las columnas:
1. entidad
2. nit_entidad
3. departamento_entidad
4. ciudad_entidad
5. ordenentidad
6. codigo_pci
7. id_del_proceso
8. referencia_del_proceso
9. ppi
10. id_del_portafolio
11. nombre_del_procedimiento
12. descripci_n_del_procedimiento
13. fase
14. fecha_de_publicacion_del
15. fecha_de_ultima_publicaci
16. fecha_de_publicacion_fase_3
17. precio_base
18. modalidad_de_contratacion
19. justificaci_n_modalidad_de
20. duracion
21. unidad_de_duracion
22. ciudad_de_la_unidad_de
23. nombre_de_la_unidad_de
24. proveedores_invitados
25. proveedores_con_invitacion
26. visualizaciones_del
27. proveedores_que_manifestaron
28. respuestas_al_procedimiento
29. respuestas_externas
30. conteo_de_respuestas_a_ofertas
31. proveedores_unicos_con
32. numero_de_lotes
33. estado_del_procedimiento
34. id_estado_del_procedimiento
35. adjudicado
36. id_adjudicacion
37. codigoproveedor
38. departamento_proveedor
39. ciudad_pro

In [2]:
# Identificar el último archivo trimestral existente
archivos = sorted([f for f in os.listdir(directorio_salida) if f.endswith(".parquet")])
if not archivos:
    raise FileNotFoundError(" No existen archivos trimestrales previos en la carpeta.")

ultimo_archivo = archivos[-1]
ruta_ultimo = os.path.join(directorio_salida, ultimo_archivo)
print(f"\n Último archivo detectado: {ultimo_archivo}")

# Leer el último archivo y obtener la última fecha
df_existing = pd.read_parquet(ruta_ultimo)
if "fecha_de_publicacion_del" not in df_existing.columns:
    raise KeyError(" No se encuentra la columna 'fecha_de_publicacion_del' en el archivo parquet.")

df_existing["fecha_de_publicacion_del"] = pd.to_datetime(
    df_existing["fecha_de_publicacion_del"], errors="coerce", utc=True
)
ultima_fecha = df_existing["fecha_de_publicacion_del"].max()
print(f"📅 Última fecha registrada en el archivo: {ultima_fecha.strftime('%Y-%m-%d')}")



 Último archivo detectado: datos_trimestre_2025_Q4_2025-10-01_to_2025-10-05.parquet
📅 Última fecha registrada en el archivo: 2025-10-05


In [3]:

# Consultar API desde la última fecha hasta hoy
fecha_inicio = (ultima_fecha + timedelta(days=1)).strftime("%Y-%m-%dT00:00:00")
fecha_fin = pd.Timestamp.today().strftime("%Y-%m-%dT23:59:59")

print(f"\n Consultando posibles nuevos registros desde {fecha_inicio} hasta {fecha_fin}...")

# Parámetros iniciales
limit = 100000   # tamaño de bloque (ajustable)
offset = 0
nuevos_datos = []

while True:
    params_actualizacion = {
        "$where": f"fecha_de_publicacion_del >= '{fecha_inicio}' AND fecha_de_publicacion_del <= '{fecha_fin}'",
        "$limit": limit,
        "$offset": offset
    }

    resp_update = requests.get(url, params=params_actualizacion)
    if resp_update.status_code != 200:
        print(f"❌ Error en la solicitud (código {resp_update.status_code})")
        break

    bloque = resp_update.json()
    if not bloque:  # Si ya no hay más registros, se detiene
        break

    nuevos_datos.extend(bloque)
    offset += limit  # Avanza al siguiente bloque
    print(f"Descargados {len(nuevos_datos):,} registros hasta ahora...")

print(f"\n Total de nuevos registros encontrados: {len(nuevos_datos):,}")

# Solo mostrar si realmente hay nuevos datos
if nuevos_datos:
    df_nuevos = pd.DataFrame(nuevos_datos)
    
    print("\n Se encontraron nuevos registros.")
    print(f" Total de columnas detectadas: {len(df_nuevos.columns)}")
    print("\n Nombres de las columnas:")
    for i, col in enumerate(df_nuevos.columns, 1):
        print(f"{i}. {col}")
else:
    print("No hay nuevos registros para actualizar.")




 Consultando posibles nuevos registros desde 2025-10-06T00:00:00 hasta 2025-10-05T23:59:59...

 Total de nuevos registros encontrados: 0
No hay nuevos registros para actualizar.


In [4]:

# Guardar datos nuevos por trimestre (crea o actualiza automáticamente)

def guardar_por_trimestres(df_existing, df_nuevos, directorio_salida):
    if df_nuevos.empty:
        print("\n No hay datos nuevos para guardar.")
        return

    # Asegurar formato de fecha
    df_nuevos["fecha_de_publicacion_del"] = pd.to_datetime(
        df_nuevos["fecha_de_publicacion_del"], errors="coerce", utc=True
    )
    df_existing["fecha_de_publicacion_del"] = pd.to_datetime(
        df_existing["fecha_de_publicacion_del"], errors="coerce", utc=True
    )

    # Combinar ambos dataframes y ordenar por fecha
    df_completo = pd.concat([df_existing, df_nuevos], ignore_index=True)
    df_completo = df_completo.sort_values("fecha_de_publicacion_del")

    # Identificar trimestres únicos en los datos nuevos
    df_nuevos["anio"] = df_nuevos["fecha_de_publicacion_del"].dt.year
    df_nuevos["trimestre"] = df_nuevos["fecha_de_publicacion_del"].dt.quarter

    trimestres_unicos = df_nuevos[["anio", "trimestre"]].drop_duplicates().sort_values(["anio", "trimestre"])

    print(f"\n Se detectaron nuevos datos en {len(trimestres_unicos)} trimestre(s):")
    print(trimestres_unicos)

    for _, fila in trimestres_unicos.iterrows():
        anio, trimestre = fila["anio"], fila["trimestre"]

        # Filtrar los datos correspondientes a ese trimestre
        df_trim = df_nuevos[
            (df_nuevos["anio"] == anio) & (df_nuevos["trimestre"] == trimestre)
        ].copy()

        # Determinar el rango de fechas del trimestre
        fecha_inicio_trim = df_trim["fecha_de_publicacion_del"].min().strftime("%Y-%m-%d")
        fecha_fin_trim = df_trim["fecha_de_publicacion_del"].max().strftime("%Y-%m-%d")

        # Nombre estándar del archivo
        nombre_archivo = f"datos_trimestre_{anio}_Q{trimestre}_{fecha_inicio_trim}_to_{fecha_fin_trim}.parquet"
        ruta_archivo = os.path.join(directorio_salida, nombre_archivo)

        if os.path.exists(ruta_archivo):
            # Si ya existe, se actualiza (anexar y eliminar duplicados)
            df_existente_trim = pd.read_parquet(ruta_archivo)
            df_actualizado = pd.concat([df_existente_trim, df_trim], ignore_index=True)
            df_actualizado = df_actualizado.drop_duplicates(subset=["id_del_proceso"], keep="last")
            df_actualizado.to_parquet(ruta_archivo, engine="pyarrow", index=False)
            print(f" Archivo actualizado: {nombre_archivo} ({len(df_trim)} registros nuevos)")
        else:
            # Crear nuevo archivo trimestral
            df_trim.to_parquet(ruta_archivo, engine="pyarrow", index=False)
            print(f" Archivo creado: {nombre_archivo} ({len(df_trim)} registros)")

    # Resumen
    ultima_fecha = df_nuevos["fecha_de_publicacion_del"].max()
    print(f"\n Actualización completada hasta la fecha {ultima_fecha.strftime('%Y-%m-%d')}")
    print(f" Archivos almacenados en: {directorio_salida}")

if nuevos_datos:
    df_nuevos = pd.DataFrame(nuevos_datos)
    guardar_por_trimestres(df_existing, df_nuevos, directorio_salida)
else:
    print("No hay nuevos registros para actualizar.")



No hay nuevos registros para actualizar.


# De que trato la Validacion

La idea es tener una actualizacion sencilla, que al final solo busque los registros que no se han actualizado y los agregue al documento.

Pero puede llegar a pasar que al final se salten unos registros, por ejemplo: El codigo esta diseñado para que lea la ultima fecha y agregue despues de eso, pero que pasaria si ese ultimo dia que se registra tuvo mas registros, se paerderian por eso el siguiente codigo lo que llegaria hacer es buscar en el trimestre que hay y que falto, para asi añadir esos registros que se nos pudieron haber pasado. 

In [ ]:
# ===============================================================
# 🔄 VALIDACIÓN Y ACTUALIZACIÓN DE LOS DOS ÚLTIMOS TRIMESTRES
# ===============================================================

import os
import requests
import pandas as pd
from datetime import datetime

# === Configuración general ===
directorio_salida = r"G:\Mi unidad\DatosAPI_Trimestral_Parquet"
dataset_id = "p6dx-8zbt"
url = f"https://www.datos.gov.co/resource/{dataset_id}.json"
chunk_size = 100000


# === Función para descargar datos desde la API por trimestre ===
def descargar_trimestre(anio, trimestre):
    fechas_trimestre = {
        3: (f"{anio}-07-01", f"{anio}-09-30"),
        4: (f"{anio}-10-01", f"{anio}-12-31"),
    }
    fecha_inicio, fecha_fin = fechas_trimestre[trimestre]

    print(f"\n📡 Descargando datos del trimestre Q{trimestre} {anio} ({fecha_inicio} a {fecha_fin})...")
    offset = 0
    dfs = []

    while True:
        params = {
            "$limit": chunk_size,
            "$offset": offset,
            "$order": "fecha_de_publicacion_del ASC",
            "$where": f"fecha_de_publicacion_del between '{fecha_inicio}' and '{fecha_fin}'"
        }
        resp = requests.get(url, params=params)
        if resp.status_code != 200:
            print(f"❌ Error {resp.status_code} en la descarga. Deteniendo...")
            break

        data = resp.json()
        if not data:
            break

        df_chunk = pd.DataFrame(data)
        if "fecha_de_publicacion_del" in df_chunk.columns:
            df_chunk["fecha_de_publicacion_del"] = pd.to_datetime(
                df_chunk["fecha_de_publicacion_del"], errors="coerce", utc=True
            )

        dfs.append(df_chunk)
        offset += chunk_size
        print(f"  ✅ Traídos {len(df_chunk):,} registros (offset {offset})")

    if dfs:
        df_full = pd.concat(dfs, ignore_index=True)
        print(f"📦 Total registros descargados para Q{trimestre} {anio}: {len(df_full):,}")
        return df_full
    else:
        print(f"⚠️ No se encontraron registros en Q{trimestre} {anio}.")
        return pd.DataFrame()


# === Función para validar, actualizar y limpiar los archivos locales ===
def validar_y_actualizar(df_api, anio, trimestre, fecha_inicio, fecha_fin):
    if df_api.empty:
        print(f"⚠️ Sin datos para validar Q{trimestre} {anio}.")
        return

    nombre_archivo = f"datos_trimestre_{anio}_Q{trimestre}_{fecha_inicio}_to_{fecha_fin}.parquet"
    ruta_archivo = os.path.join(directorio_salida, nombre_archivo)

    # Si el archivo no existe, crear uno nuevo
    if not os.path.exists(ruta_archivo):
        print(f"🆕 Archivo no encontrado. Creando nuevo: {nombre_archivo}")
        df_api.to_parquet(ruta_archivo, engine="pyarrow", index=False)
    else:
        # Leer el archivo existente
        df_local = pd.read_parquet(ruta_archivo)
        print(f"📂 Archivo encontrado: {nombre_archivo} (Registros: {len(df_local):,})")

        # Asegurar formato datetime
        if "fecha_de_publicacion_del" in df_local.columns:
            df_local["fecha_de_publicacion_del"] = pd.to_datetime(
                df_local["fecha_de_publicacion_del"], errors="coerce", utc=True
            )
        if "fecha_de_publicacion_del" in df_api.columns:
            df_api["fecha_de_publicacion_del"] = pd.to_datetime(
                df_api["fecha_de_publicacion_del"], errors="coerce", utc=True
            )

        # Filtrar columnas válidas
        columnas_validas = []
        for col in df_api.columns:
            muestra = df_api[col].dropna().head(5)
            if not any(isinstance(x, (dict, list)) for x in muestra):
                columnas_validas.append(col)

        cols_comunes = list(set(df_local.columns).intersection(columnas_validas))

        # Unir y eliminar duplicados
        df_union = pd.concat([df_local, df_api], ignore_index=True)
        df_union = df_union.drop_duplicates(subset=cols_comunes, keep="first")

        nuevos = len(df_union) - len(df_local)
        if nuevos > 0:
            print(f"✅ Archivo actualizado con {nuevos:,} nuevos registros.")
        else:
            print("🔹 No se encontraron nuevos registros únicos para agregar.")

        # Guardar archivo actualizado
        df_union.to_parquet(ruta_archivo, engine="pyarrow", index=False)

    # === 🔧 ELIMINAR COLUMNAS 'anio' Y 'trimestre' SI EXISTEN ===
    try:
        df_final = pd.read_parquet(ruta_archivo)
        cols_a_eliminar = [c for c in ['anio', 'trimestre'] if c in df_final.columns]
        if cols_a_eliminar:
            df_final = df_final.drop(columns=cols_a_eliminar)
            df_final.to_parquet(ruta_archivo, engine="pyarrow", index=False)
            print(f"🧹 Columnas eliminadas del archivo {nombre_archivo}: {cols_a_eliminar}")
        else:
            print("✅ No había columnas 'anio' ni 'trimestre' para eliminar.")
    except Exception as e:
        print(f"⚠️ Error eliminando columnas en {nombre_archivo}: {e}")


# === Ejecución principal ===
trimestres_objetivo = [
    (2025, 3, "2025-07-01", "2025-09-30"),  # Q3
    (2025, 4, "2025-10-01", datetime.today().strftime("%Y-%m-%d"))  # Q4
]

for anio, trimestre, fecha_inicio, fecha_fin in trimestres_objetivo:
    print(f"\n📆 Procesando Q{trimestre} {anio} ({fecha_inicio} a {fecha_fin})...")
    df_api_trim = descargar_trimestre(anio, trimestre)
    validar_y_actualizar(df_api_trim, anio, trimestre, fecha_inicio, fecha_fin)

print("\n🎯 Validación completa de los dos últimos trimestres.")



📆 Procesando Q3 2025 (2025-07-01 a 2025-09-30)...

📡 Descargando datos del trimestre Q3 2025 (2025-07-01 a 2025-09-30)...
  ✅ Traídos 100,000 registros (offset 100000)
  ✅ Traídos 100,000 registros (offset 200000)


In [2]:
import os
import shutil
notebook_actual = "Actualizacion_de_Datos.ipynb"
ruta_destino = r"G:\Mi unidad\Colab Notebooks"

os.makedirs(ruta_destino, exist_ok=True)
shutil.copy(notebook_actual, os.path.join(ruta_destino, notebook_actual))

print(f"✅ Notebook guardado en {ruta_destino}")

✅ Notebook guardado en G:\Mi unidad\Colab Notebooks
